# 03 — GPT-4o Vision smoke test

Sends an image (URL) to GPT-4o vision and prints the description.
Drop any image at `notebooks/sample.png`.

In [3]:
import sys, os, pathlib
sys.path.insert(0, os.path.abspath('..'))
from src.blob_client import BlobService
from src.openai_client import OpenAIService

blob = BlobService()
blob.ensure_container()

IMG = pathlib.Path('sample.png')
if IMG.exists():
    # Use local file
    name = 'docmind-test/vision-sample.png'
    blob.upload(name, IMG.read_bytes(), content_type='image/png')
    url = blob.get_url(name)
    print('Using local sample.png')
else:
    # Fall back to the first image extracted by notebook 02
    candidates = [
        b for b in blob.list_blobs('docmind-test/')
        if b.endswith(('.png', '.jpg', '.jpeg', '.webp'))
        and ('/images/' in b or '/figures/' in b)
    ]
    assert candidates, (
        'No extracted images found in blob — run notebook 02 first, '
        'or drop a sample.png next to this notebook.'
    )
    name = candidates[0]
    url = blob.get_url(name)
    print(f'Using blob image from notebook 02: {name}')

print('Image URL:', url[:80], '…')

INFO: Request URL: 'https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input?restype=REDACTED&sp=REDACTED&st=REDACTED&se=REDACTED&spr=REDACTED&sv=REDACTED&sr=REDACTED&sig=REDACTED'
Request method: 'PUT'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': 'c3a6fa07-4a51-11f1-9614-bcd22c162bea'
No body was attached to the request
INFO: Response status: 403
Response headers:
    'Content-Length': '246'
    'Content-Type': 'application/xml'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': '7a3aba6a-b01e-009f-0b5e-de19ff000000'
    'x-ms-client-request-id': 'c3a6fa07-4a51-11f1-9614-bcd22c162bea'
    'x-ms-version': 'REDACTED'
    'x-ms-error-code': 'AuthorizationFailure'
    'Date': 'Thu, 07 May 2026 20:17:26 GMT'
INFO: Request URL: 'https://pdgrag.blob.core.windows.net/sgp-d

Using blob image from notebook 02: docmind-test/figures/page15_fig0_0.png
Image URL: https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-test/figures/ …


In [4]:
ai = OpenAIService()
desc = ai.describe_image(url)
print(desc)

INFO: HTTP Request: POST https://azr-oai-dai-sandbox4-001.openai.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-08-01-preview "HTTP/1.1 200 OK"


This image is a flowchart depicting a process for handling user queries through a structured task management system. Below is a detailed description of the components and their relationships:

1. **User Query**: The process begins with a user query.

2. **Planner Node**: The query is sent to the Planner Node.

3. **Analyze Intent + Define Goal**: The intent of the query is analyzed, and a goal is defined.

4. **Research Intent Classification**: The intent is classified into categories such as blog, comparative, deep research, or summary.

5. **Structured Plan Creation**: A structured plan is created based on the classification.

6. **Generate Todos with status=pending**: Tasks (Todos) are generated with a pending status.

7. **Task Manager**: Manages the tasks and sends them to the Todo Selector.

8. **Todo Selector**: Selects pending tasks for processing.

9. **Delegation Layer**: Distributes tasks to multiple sub-agents.

10. **Sub-Agents (1, 2, N)**: Each sub-agent works on tasks an

In [5]:
# Bonus — test embeddings round-trip
v = ai.embed('Hello DocMind')[0]
print('embedding dim:', len(v))

INFO: HTTP Request: POST https://azr-oai-dai-sandbox4-001.openai.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-08-01-preview "HTTP/1.1 200 OK"


embedding dim: 1536
